## Sentiment Analysis and Model Evaluation

Inspired by Géron’s (2022) discussion of NLP and Hugging Face, this notebook uses a pretrained DistilBERT sentiment classifier to explore prediction performance.

My **sentiment-classification** exercise analysed **accuracy, precision, recall, F1 and a confusion matrix**.
The model interpreted the initial straightforward sentences correctly and achieved 100% accuracy. However, when it encountered more ambiguity and contrasting opinions, accuracy dropped to 70%. This drop demonstrated that high performance on clear sentences does not guarantee success on more ambiguous examples. Furthermore, the confusion matrix shows that negative-class recall was only 40%. This confirms that accuracy alone is insufficient for evaluating a classifier. The confusion matrix and class-level metrics revealed errors that the overall accuracy score concealed.

* **Accuracy:** proportion of predictions that were correct (match the labels) *==> (correct predictions / all predictions)*
* **Precision:** when the model predicts a class, how often it is correct *==> (Correct predictions of that class / all predictions of that class)*
* **Recall (Sensitivity):** Of all examples actually belonging to a class, how many did the model identify? *==> (Correct predictions of that class / all actual examples of that class)*
* **Specificity:** Of all actual negative examples, how many did the model correctly identify as negative? *==> (Correct negative predictions / all actual negative examples)*

**Precision checks the predictions I made and recall checks the examples I missed.**

* **F1:** combines precision and recall.

In [1]:
#Load model DistilBERT
from transformers import pipeline

sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=-1
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### TEST Labelled SET 1 

In [2]:
# Create a small labelled test set with expected labels (SET 1)

texts = [
    "I loved this film. The acting was wonderful.",
    "The story was engaging and beautifully told.",
    "An excellent film that I would happily watch again.",
    "It started slowly, but I enjoyed it overall.",
    "I expected to hate it, but it was surprisingly good.",
    "I hated this film. It was boring.",
    "The acting was dreadful and the story made no sense.",
    "A disappointing film that wasted my time.",
    "It looked beautiful, but overall it was terrible.",
    "I expected to enjoy it, but it was painfully dull."
]

y_true = [
    "POSITIVE", "POSITIVE", "POSITIVE", "POSITIVE", "POSITIVE",
    "NEGATIVE", "NEGATIVE", "NEGATIVE", "NEGATIVE", "NEGATIVE"
]

In [3]:
# Predict and inspect (SET 1)
results = sentiment(texts)
y_pred = [result["label"] for result in results]

for text, actual, result in zip(texts, y_true, results):
    print(text)
    print(
        f"Expected: {actual} | Predicted: {result['label']} | "
        f"Confidence: {result['score']:.3f}"
    )
    print()

I loved this film. The acting was wonderful.
Expected: POSITIVE | Predicted: POSITIVE | Confidence: 1.000

The story was engaging and beautifully told.
Expected: POSITIVE | Predicted: POSITIVE | Confidence: 1.000

An excellent film that I would happily watch again.
Expected: POSITIVE | Predicted: POSITIVE | Confidence: 1.000

It started slowly, but I enjoyed it overall.
Expected: POSITIVE | Predicted: POSITIVE | Confidence: 1.000

I expected to hate it, but it was surprisingly good.
Expected: POSITIVE | Predicted: POSITIVE | Confidence: 1.000

I hated this film. It was boring.
Expected: NEGATIVE | Predicted: NEGATIVE | Confidence: 1.000

The acting was dreadful and the story made no sense.
Expected: NEGATIVE | Predicted: NEGATIVE | Confidence: 1.000

A disappointing film that wasted my time.
Expected: NEGATIVE | Predicted: NEGATIVE | Confidence: 1.000

It looked beautiful, but overall it was terrible.
Expected: NEGATIVE | Predicted: NEGATIVE | Confidence: 0.995

I expected to enjoy it,

In [4]:
## Calculate evaluation metrics
from sklearn.metrics import accuracy_score, classification_report

print(f"Accuracy: {accuracy_score(y_true, y_pred):.1%}")

print(classification_report(
    y_true,
    y_pred,
    labels=["POSITIVE", "NEGATIVE"],
    zero_division=0
))

Accuracy: 100.0%
              precision    recall  f1-score   support

    POSITIVE       1.00      1.00      1.00         5
    NEGATIVE       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



In [5]:
# Display confision matrix 
from sklearn.metrics import confusion_matrix

matrix = confusion_matrix(
    y_true,
    y_pred,
    labels=["POSITIVE", "NEGATIVE"]
)

print(matrix)

[[5 0]
 [0 5]]


The model correctly predicted labels for all sentences and achieved 100% accuracy.

### TEST Labelled SET 2 (introducing ambiguity)

In [6]:
# Create a small labelled test set with expected labels (SET 2)

texts = [
    "I loved this film. The acting was wonderful.",
    "It wasn't perfect, but I would happily watch it again.",
    "I expected to hate it, but it was surprisingly good.",
    "The beginning was dull, but overall I really enjoyed it.",
    "Despite its flaws, this film was well worth watching.",
    "I hated this film. It was boring.",
    "Wonderful—two hours of my life I'll never get back.",
    "The best part of this film was when it finally ended.",
    "The acting was excellent, but overall I disliked the film.",
    "I wouldn't recommend it, even though a few scenes were good."
]

y_true = [
    "POSITIVE", "POSITIVE", "POSITIVE", "POSITIVE", "POSITIVE",
    "NEGATIVE", "NEGATIVE", "NEGATIVE", "NEGATIVE", "NEGATIVE"
]

In [7]:
# Predict and inspect
results = sentiment(texts)
y_pred = [result["label"] for result in results]

for text, actual, result in zip(texts, y_true, results):
    print(text)
    print(
        f"Expected: {actual} | Predicted: {result['label']} | "
        f"Confidence: {result['score']:.3f}"
    )
    print()

I loved this film. The acting was wonderful.
Expected: POSITIVE | Predicted: POSITIVE | Confidence: 1.000

It wasn't perfect, but I would happily watch it again.
Expected: POSITIVE | Predicted: POSITIVE | Confidence: 1.000

I expected to hate it, but it was surprisingly good.
Expected: POSITIVE | Predicted: POSITIVE | Confidence: 1.000

The beginning was dull, but overall I really enjoyed it.
Expected: POSITIVE | Predicted: POSITIVE | Confidence: 1.000

Despite its flaws, this film was well worth watching.
Expected: POSITIVE | Predicted: POSITIVE | Confidence: 1.000

I hated this film. It was boring.
Expected: NEGATIVE | Predicted: NEGATIVE | Confidence: 1.000

Wonderful—two hours of my life I'll never get back.
Expected: NEGATIVE | Predicted: POSITIVE | Confidence: 1.000

The best part of this film was when it finally ended.
Expected: NEGATIVE | Predicted: POSITIVE | Confidence: 0.995

The acting was excellent, but overall I disliked the film.
Expected: NEGATIVE | Predicted: NEGATIVE 

In [8]:
# Display confision matrix
from sklearn.metrics import confusion_matrix

matrix = confusion_matrix(
    y_true,
    y_pred,
    labels=["POSITIVE", "NEGATIVE"]
)

print(matrix)

[[5 0]
 [3 2]]


* The matrix shows that 7 of 10 sentences were classified correctly. 
* All 5 positive sentences were recognised, but only 2 of 5 negative sentences were correctly identified. 
* The remaining 3 sentences were incorrectly predicted as positive

In [9]:
## Calculate evaluation metrics
from sklearn.metrics import accuracy_score, classification_report

print(f"Accuracy: {accuracy_score(y_true, y_pred):.1%}")

print(classification_report(
    y_true,
    y_pred,
    labels=["POSITIVE", "NEGATIVE"],
    zero_division=0
))

Accuracy: 70.0%
              precision    recall  f1-score   support

    POSITIVE       0.62      1.00      0.77         5
    NEGATIVE       1.00      0.40      0.57         5

    accuracy                           0.70        10
   macro avg       0.81      0.70      0.67        10
weighted avg       0.81      0.70      0.67        10



- **Accuracy:** 70%, 7 of 10 ten sentences were matched the expected labels. 
- **Positive recall** = 1.00: the model found all 5 positive examples.
- **Positive precision** = 0.62: only 5 of its 8 positive predictions were correct.
- **Negative recall** = 0.40: the model identified only 2 of 5 negative examples.
- **Negative precision** = 1.00: both negative predictions were correct.

